# 🧚 동화 AI — Google Colab 학습 노트북

| 단계 | 내용 | 시간 |
|------|------|------|
| 1 | GPU 확인 & 라이브러리 설치 | ~5분 |
| 2 | Google Drive 마운트 | 1분 |
| 3 | 데이터 로드 | ~30초 |
| 4 | 모델 & LoRA 설정 | ~10분 |
| 5 | 학습 | ~30~50분 |
| 6 | 저장 & 테스트 | ~10분 |

> 💡 상단 메뉴: `런타임` → `런타임 유형 변경` → `T4 GPU` 선택

## ✅ 1단계 — GPU 확인 & 라이브러리 설치

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
!pip install -q bitsandbytes>=0.45.0
!pip install -q transformers>=4.40.0 peft>=0.10.0 datasets accelerate sentencepiece protobuf

import transformers, peft, bitsandbytes
print('\n✅ 설치 완료!')
print(f'  transformers: {transformers.__version__}')
print(f'  peft:         {peft.__version__}')
print(f'  bitsandbytes: {bitsandbytes.__version__}')

## ✅ 2단계 — Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/fairytale_ai'
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/final_model', exist_ok=True)
print(f'✅ Drive 마운트 완료: {WORK_DIR}')

## ✅ 3단계 — 데이터 로드

In [ ]:
import os, json, random
from datasets import Dataset

# 경로 직접 지정
train_path = '/content/drive/MyDrive/fairytale_ai/train.jsonl'
val_path   = '/content/drive/MyDrive/fairytale_ai/val.jsonl'

# 풀학습: 전체 데이터 사용
QUICK_TEST  = False
SAMPLE_SIZE = 1000

def load_jsonl(path, n=None):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    if n and len(rows) > n:
        random.seed(42)
        rows = random.sample(rows, n)
    return Dataset.from_list(rows)

train_ds = load_jsonl(train_path, SAMPLE_SIZE if QUICK_TEST else None)
val_ds   = load_jsonl(val_path,   150 if QUICK_TEST else None)

print(f'✅ 학습: {len(train_ds):,}개 / 검증: {len(val_ds):,}개')
print(train_ds[0]['text'][:200])

## ✅ 4단계 — 모델 & LoRA 설정

In [ ]:
import os, torch, gc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'   # 3B (풀학습 권장)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'모델 로드 중: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                  # 풀학습: 16
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
vram = torch.cuda.memory_allocated(0)/1024**3
print(f'\n✅ 준비 완료!')
print(f'   학습 파라미터: {trainable:,} ({100*trainable/total:.2f}%)')
print(f'   VRAM 사용: {vram:.1f} GB')

## ✅ 5단계 — 학습

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

MAX_LEN = 768   # 3B + 3시간 최적화

def tokenize_fn(examples):
    out = tokenizer(examples['text'], truncation=True, max_length=MAX_LEN, padding='max_length')
    out['labels'] = out['input_ids'].copy()
    return out

print('토크나이즈 중...')
train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=['text'])
val_tok   = val_ds.map(tokenize_fn,   batched=True, remove_columns=['text'])
print(f'✅ 완료! 학습: {len(train_tok):,}개 / 검증: {len(val_tok):,}개')

In [ ]:
OUTPUT_DIR = f'{WORK_DIR}/checkpoints'

# 3시간 내 완료 최적화 설정 (3B 모델 / T4 기준)
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,               # 2에폭 (~3시간)
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,   # 실질 배치=16
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    fp16=True,
    optim='paged_adamw_8bit',
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
    dataloader_num_workers=0,
    remove_unused_columns=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

steps = len(train_tok) // 16
print(f'✅ Trainer 준비 완료!')
print(f'   전체 데이터: {len(train_tok):,}개')
print(f'   예상 스텝: ~{steps * 2:,} (2에폭)')
print(f'   예상 시간: ~2.5~3.5시간')

In [ ]:
# 🚀 학습 시작!
trainer.train()
print('\n🎉 학습 완료!')

## ✅ 6단계 — 저장 & 테스트

In [ ]:
FINAL_DIR = f'{WORK_DIR}/final_model'
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(FINAL_DIR) for f in files
) / 1024**2
print(f'✅ 저장 완료! ({size_mb:.0f} MB) → {FINAL_DIR}')

In [ ]:
import torch

def generate(prompt, max_new_tokens=300):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# 동화 생성 테스트
print('='*50)
print('📖 동화 생성 테스트')
print('='*50)
story = generate('<s>[INST] 장르: 판타지, 대상 연령: 유아\n작은 토끼가 마법의 당근을 발견하는 이야기를 써줘 [/INST]')
print(story)

print('\n' + '='*50)
print('🎭 선택지 생성 테스트')
print('='*50)
choices = generate(f'<s>[INST] 다음 동화의 선택지 3개를 만들어주세요.\n\n{story[:200]} [/INST]', 150)
print(choices)

## 🎉 완료!

### 빠른 테스트 → 풀 학습 전환

| 설정 | 빠른 테스트 | 풀 학습 |
|------|------------|----------|
| `QUICK_TEST` | `True` | `False` |
| `num_train_epochs` | `1` | `3` |
| `r` (LoRA rank) | `8` | `16` |
| `MAX_LEN` | `512` | `1024` |
| 예상 시간 | ~30~50분 | ~3~6시간 |